In [ ]:
import random
import numpy as np
import time
import matplotlib.pyplot as plt

# Helper functions to read TSP files and calculate distance
def read_tsp_file(file_path):
    with open(file_path, 'r') as f:
        lines = f.readlines()

    coords = []
    reading_coords = False

    for line in lines:
        # Start reading coordinates after the 'NODE_COORD_SECTION' header
        if 'NODE_COORD_SECTION' in line:
            reading_coords = True
            continue
        if 'EOF' in line:  # End of data section
            break
        if reading_coords:
            # Only parse lines with coordinates (skip other lines)
            try:
                parts = line.split()
                if len(parts) >= 3:  # Ensure there are at least 3 items: ID, x, and y
                    x, y = float(parts[1]), float(parts[2])
                    coords.append((x, y))
            except ValueError:
                continue  # Skip lines that can't be parsed (e.g., empty or invalid lines)

    return coords


def calculateDistance(tour, distance_matrix):
    total_distance = 0.0
    for i in range(len(tour) - 1):
        total_distance += distance_matrix[tour[i]][tour[i + 1]]
    total_distance += distance_matrix[tour[-1]][tour[0]]  # Return to the start
    return total_distance

# PSO Algorithm for TSP
def pso(coords):
    n = len(coords)
    # Create the distance matrix
    distance_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i != j:
                distance_matrix[i][j] = np.linalg.norm(np.array(coords[i]) - np.array(coords[j]))

    # PSO Parameters
    num_particles = 50
    max_iter = 100
    inertia_weight = 0.7
    cognitive_coeff = 1.5
    social_coeff = 1.5

    # Initialize particles (each particle is a random permutation of city indices)
    particles = [list(np.random.permutation(n)) for _ in range(num_particles)]
    velocities = [np.zeros(n) for _ in range(num_particles)]
    personal_best = particles[:]
    personal_best_scores = [calculateDistance(p, distance_matrix) for p in personal_best]

    # Global best
    global_best = personal_best[np.argmin(personal_best_scores)]
    global_best_score = min(personal_best_scores)

    # Main PSO loop
    for iteration in range(max_iter):
        for i, particle in enumerate(particles):
            # Calculate fitness
            fitness = calculateDistance(particle, distance_matrix)

            # Update personal best
            if fitness < personal_best_scores[i]:
                personal_best[i] = particle[:]
                personal_best_scores[i] = fitness

            # Update global best
            if fitness < global_best_score:
                global_best = particle[:]
                global_best_score = fitness

            # Update velocity and position
            velocities[i] = inertia_weight * velocities[i] + cognitive_coeff * random.random() * (np.array(personal_best[i]) - np.array(particle)) + social_coeff * random.random() * (np.array(global_best) - np.array(particle))
            particles[i] = list(np.array(particle) + velocities[i])

            # Ensure the particle's position is a valid permutation of city indices
            particles[i] = list(np.argsort(particles[i]))

        print(f"Iteration {iteration+1}/{max_iter}, Best Distance: {global_best_score}")

    return global_best_score, global_best

# Glowworm Swarm Optimization (GSO) for TSP
def gso(coords):
    n = len(coords)
    distance_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i != j:
                distance_matrix[i][j] = np.linalg.norm(np.array(coords[i]) - np.array(coords[j]))

    NUM_GLOWWORMS = 50
    MAX_ITERATIONS = 100
    GAMMA = 0.6

    class Glowworm:
        def __init__(self):
            self.tour = []
            self.brightness = 0.0

    def calculateDistance(tour):
        totalDistance = 0.0
        for i in range(len(tour) - 1):
            totalDistance += distance_matrix[tour[i]][tour[i + 1]]
        totalDistance += distance_matrix[tour[-1]][tour[0]]  # Return to starting city
        return totalDistance

    def initializeGlowworms():
        glowworms = []
        for i in range(NUM_GLOWWORMS):
            newGlowworm = Glowworm()
            newGlowworm.tour = list(range(len(distance_matrix)))

            # Initialize tour with cities in order
            random.shuffle(newGlowworm.tour[1:])  # Shuffle cities except starting city
            newGlowworm.brightness = 1.0 / calculateDistance(newGlowworm.tour)
            glowworms.append(newGlowworm)
        return glowworms

    def swapTwoCities(tour, city1, city2):
        tour[city1], tour[city2] = tour[city2], tour[city1]

    glowworms = initializeGlowworms()

    for iter in range(MAX_ITERATIONS):
        # Update glowworm positions and brightness
        totalBrightness = sum(gw.brightness for gw in glowworms)
        for i in range(NUM_GLOWWORMS):
            # Glowworm Movement
            decisionRange = GAMMA * glowworms[i].brightness / totalBrightness
            neighborIndices = []

            for j in range(NUM_GLOWWORMS):
                if i != j and calculateDistance(glowworms[j].tour) < calculateDistance(glowworms[i].tour) + decisionRange:
                    neighborIndices.append(j)

            if neighborIndices:
                selectedNeighborIndex = random.choice(neighborIndices)
                city1 = random.randint(1, len(glowworms[i].tour) - 1)  # Exclude the starting city
                city2 = random.randint(1, len(glowworms[selectedNeighborIndex].tour) - 1)
                swapTwoCities(glowworms[i].tour, city1, city2)
                glowworms[i].brightness = 1.0 / calculateDistance(glowworms[i].tour)

            # Local Search (2-opt)
            bestTour = glowworms[i].tour
            bestDistance = calculateDistance(bestTour)
            for j in range(len(bestTour) - 1):
                for k in range(j + 1, len(bestTour)):
                    newTour = bestTour[:]
                    newTour[j:k+1] = reversed(newTour[j:k+1])
                    newDistance = calculateDistance(newTour)
                    if newDistance < bestDistance:
                        bestTour = newTour
                        bestDistance = newDistance
            glowworms[i].tour = bestTour
            glowworms[i].brightness = 1.0 / bestDistance

        glowworms.sort(key=lambda x: x.brightness, reverse=True)

    bestGlowworm = max(glowworms, key=lambda gw: gw.brightness)
    return calculateDistance(bestGlowworm.tour), bestGlowworm.tour

# Genetic Algorithm for TSP
def genetic_algorithm(coords):
    n = len(coords)
    distance_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i != j:
                distance_matrix[i][j] = np.linalg.norm(np.array(coords[i]) - np.array(coords[j]))

    POP_SIZE = 100
    MAX_ITER = 100
    MUTATION_RATE = 0.1
    CROSSOVER_RATE = 0.8

    def create_individual():
        return list(np.random.permutation(n))

    def fitness(individual):
        return calculateDistance(individual, distance_matrix)

    def crossover(parent1, parent2):
        start, end = sorted(random.sample(range(n), 2))
        child = [-1] * n
        child[start:end+1] = parent1[start:end+1]
        pointer = 0
        for i in range(n):
            if child[i] == -1:
                while parent2[pointer] in child:
                    pointer += 1
                child[i] = parent2[pointer]
        return child

    def mutate(individual):
        if random.random() < MUTATION_RATE:
            i, j = random.sample(range(n), 2)
            individual[i], individual[j] = individual[j], individual[i]

    population = [create_individual() for _ in range(POP_SIZE)]

    for gen in range(MAX_ITER):
        population = sorted(population, key=fitness)
        next_generation = population[:2]

        while len(next_generation) < POP_SIZE:
            if random.random() < CROSSOVER_RATE:
                parent1, parent2 = random.sample(population[:50], 2)
                child = crossover(parent1, parent2)
            else:
                child = random.choice(population[:50])
            mutate(child)
            next_generation.append(child)

        population = next_generation

    best_individual = min(population, key=fitness)
    return fitness(best_individual), best_individual

# Main function
def main():
    datasets = ['burma14.tsp', 'berlin52.tsp', 'bier127.tsp']  # List of datasets
    results = []

    for path in datasets:
        coords = read_tsp_file(path)
        print(f"\nRunning on {path} with {len(coords)} cities")

        # Run the algorithms
        d1, _ = pso(coords)
        d2, _ = gso(coords)
        d3, _ = genetic_algorithm(coords)

        # Store results
        results.append((path, d1, d2, d3))
        print(f"PSO Best Distance: {d1}")
        print(f"GSO Best Distance: {d2}")
        print(f"Genetic Algorithm Best Distance: {d3}")

    # Output results for comparison
    print("\nComparison of Best Distances:")
    for result in results:
        print(f"{result[0]}: PSO={result[1]}, GSO={result[2]}, GA={result[3]}")

if __name__ == "__main__":
    main()



Running on burma14.tsp with 14 cities
Iteration 1/100, Best Distance: 47.67368296333787
Iteration 2/100, Best Distance: 46.905071500248276
Iteration 3/100, Best Distance: 45.67025096561539
Iteration 4/100, Best Distance: 45.67025096561539
Iteration 5/100, Best Distance: 41.8616141023712
Iteration 6/100, Best Distance: 41.8616141023712
Iteration 7/100, Best Distance: 41.8616141023712
Iteration 8/100, Best Distance: 41.8616141023712
Iteration 9/100, Best Distance: 41.8616141023712
Iteration 10/100, Best Distance: 41.8616141023712
Iteration 11/100, Best Distance: 41.8616141023712
Iteration 12/100, Best Distance: 41.8616141023712
Iteration 13/100, Best Distance: 41.8616141023712
Iteration 14/100, Best Distance: 41.8616141023712
Iteration 15/100, Best Distance: 41.8616141023712
Iteration 16/100, Best Distance: 41.8616141023712
Iteration 17/100, Best Distance: 41.8616141023712
Iteration 18/100, Best Distance: 40.69078614663144
Iteration 19/100, Best Distance: 40.69078614663144
Iteration 20/